# TMalign Structure Alignment

![TMalign Structure Alignment](https://proto-bio.github.io/proto-assets/images/tool/tmalign/hero.png)

This notebook demonstrates pairwise protein structure alignment using TMalign. We align two structures provided as PDB text and inspect the TM-scores, which provide a length-independent measure of topological similarity between the two inputs.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("tmalign")
display_overview("tmalign")
display_docs_section("tmalign", "Background")

# TMalign

[TMalign](https://zhanggroup.org/TM-align/) is a pairwise protein structure alignment program developed by the [Zhang Lab](https://zhanggroup.org/). It identifies the optimal structural superposition of two protein structures by directly maximising the Template Modeling score (TM-score), and reports the score normalised by the length of each input chain. This toolkit compiles TMalign from the canonical [pylelab/USalign](https://github.com/pylelab/USalign) distribution and runs it through a single registered tool that returns both length-normalised TM-scores.

The Template Modeling score (TM-score) ([Zhang and Skolnick, 2004](https://doi.org/10.1002/prot.20264)) is a length-independent measure of topological similarity between two protein structures. It scores each pair of corresponding residues with a distance-based weight that uses a protein-size-dependent normalisation, which eliminates the inherent length dependence of RMSD-style scores and lets the same TM-score value be compared across proteins of different sizes. The score ranges from 0 to 1, with 1 indicating identical structures.

TMalign ([Zhang and Skolnick, 2005](https://doi.org/10.1093/nar/gki524)) is a structure alignment algorithm that identifies the optimal pairwise structural superposition by combining a TM-score-based rotation matrix with dynamic programming. Three initial alignments are seeded from secondary-structure matching, gapless threading, and a hybrid scoring matrix, and the residue-to-residue correspondence is then iteratively refined by alternating rigid-body rotation with dynamic programming on the TM-score-weighted distance matrix until the alignment converges. Unlike alignment methods that optimise RMSD, TMalign directly optimises the TM-score, which decouples the alignment objective from chain length. The published benchmark reports that TMalign produces alignments with higher coverage and accuracy than CE, DALI, and SAL while running approximately four times faster than CE and twenty times faster than DALI on the same workload.

A subsequent statistical analysis of the TM-score ([Xu and Zhang, 2010](https://doi.org/10.1093/bioinformatics/btq066)) provides quantitative interpretation guidance. The authors compare TM-scores across all pairs in a non-redundant set of 6,684 single-domain protein structures and report that the score follows an extreme value distribution. They show that a TM-score above 0.5 is a strong probabilistic indicator of shared SCOP and CATH fold classification, while scores below 0.5 mostly indicate different folds.

## Available tools

In [2]:
display_available_tools("tmalign")

- **`run_tmalign()`** — Pairwise protein structure alignment using TMalign (Zhang & Skolnick, 2005). Returns TM-scores normalized by each chain.

### `run_tmalign`

TMalign performs pairwise structure alignment by iteratively superimposing two protein structures to maximize the TM-score. It accepts two structures as raw PDB text, writes them to temporary files, calls the TMalign binary, and returns two TM-scores — one normalized by the length of each chain. The score normalized by the reference chain is typically used to evaluate whether a candidate structure matches a target fold.

In [3]:
from pathlib import Path

from proto_tools import Structure, TMalignConfig, TMalignInput, TMalignOutput, run_tmalign

In [4]:
# Display input docs
display_api_reference("tmalign", "input", "run_tmalign")

# Align two homologous TIM-barrel structures (1TIM, 8TIM) fetched from the RCSB,
# so the TM-score reflects a real fold comparison rather than self-alignment.
query = Structure.from_rcsb("1TIM")
reference = Structure.from_rcsb("8TIM")
inputs = TMalignInput(query_structure=query, reference_structure=reference)

**Input** — `TMalignInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query_structure</code> | <code>Structure</code> | required | Query / candidate structure (Structure object, file path, or raw PDB/CIF string) |
| <code>reference_structure</code> | <code>Structure</code> | required | Reference / target structure (Structure object, file path, or raw PDB/CIF string) |

In [5]:
# Display config docs
display_api_reference("tmalign", "config", "run_tmalign")

# TMalignConfig has no tool-specific parameters; all defaults are used
config = TMalignConfig()

**Config** — `TMalignConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |

In [6]:
# Run the alignment
result = run_tmalign(inputs, config)

Running run_tmalign [00:00]

In [7]:
# Display output docs and inspect results
display_api_reference("tmalign", "output", "run_tmalign")

print(f"TM-score (norm by Chain 1): {result.tm_score_chain_1:.3f}")
print(f"TM-score (norm by Chain 2): {result.tm_score_chain_2:.3f}")

**Output** — `TMalignOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>metrics</code> | <code>TMalignMetrics</code> | <code>TMalignMetrics(primary_metric=None, metric_type='TMalignMetrics')</code> | Pairwise alignment scores |

**Metrics** — `TMalignMetrics`

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>tm_score_chain_1</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>tm_score_chain_2</code> | <code>float</code> | <code>[0, 1]</code> |  |  |

TM-score (norm by Chain 1): 0.980
TM-score (norm by Chain 2): 0.980


## Export Results

Alignment results can be exported to JSON format for downstream analysis or integration with other computational pipelines.

In [8]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("tmalign_result", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'tmalign_result.json'}")

Exported to example_output/tmalign_result.json
